# Original sccGAN + natural-E4.5-calibrated critic scoring

这个 notebook 保留 `sccGAN code.ipynb` 中的 GAN 网络结构以及原始 `critic.pth` 权重，只更新评分和校准方法。它用于回答两个不同的问题：

1. **同谱系相似度**：EPI-like 对 Pre_EPI；PE-like 分别对 Pre_VE、PaE；TE-like 分别对 mTE、pTE。
2. **错误标签测试**：把每个已注释谱系同时送入五个条件，检查跨谱系条件是否产生更大的距离。

原始 WGAN critic 输出是无界实数，并不是概率，也不是可以跨数据集直接比较的距离。因此不再直接比较 natural E4.5 的 10–20 与 blastoid 的 0.x，而是计算：

$$D(q,c)=\frac{W_1(\{C(x,c):x\in q\},\{C(x,c):x\in E4.5_{cal,c}\})}{\operatorname{IQR}(C(E4.5_{cal,c},c))/1.349}$$

其中 $c$ 是自然 E4.5 条件，$q$ 是待评价细胞群。**最终距离越低，表示其 critic 输出分布越接近相应的自然 E4.5 校准分布。** 95% CI 由细胞重采样 bootstrap 得到。

> 注意：natural E4.5 的 70/30 划分用于评分校准与内部 holdout 检查；如果 `E450_expression.csv` 来自同一批自然细胞，这不是独立于 GAN 训练数据的外部验证集。

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
DATA = ROOT / 'data'
SCRIPTS = ROOT / 'scripts'
CHECKPOINTS = ROOT / 'checkpoints'
PREPARED = ROOT / 'prepared'
RESULTS = ROOT / 'results' / 'calibrated'
EXAMPLE_RESULTS = ROOT / 'results' / 'example'
ACTIVE_RESULTS = RESULTS if (RESULTS / 'overall_distances.csv').exists() else EXAMPLE_RESULTS
CHECKPOINT = CHECKPOINTS / 'critic.pth'

rscript = shutil.which('Rscript')
if rscript is None:
    candidate = Path(r'C:\Program Files\R\R-4.4.2\bin\Rscript.exe')
    rscript = str(candidate) if candidate.exists() else None

required = [
    DATA / 'E450_expression.csv', DATA / 'E4.5.rds', CHECKPOINT,
    SCRIPTS / 'prepare_inputs.R', SCRIPTS / 'score_critic.py',
]
missing = [str(path) for path in required if not path.exists()]
print('Working directory:', ROOT)
print('Python:', sys.executable)
print('Rscript:', rscript)
print('Missing required files:', missing if missing else 'none')

## 1. Optional: rebuild the common fixed-depth inputs

Natural E4.5 and every blastoid dataset must enter the critic on the same scale and in the exact 2,000-gene order used for training. The preprocessing script:

- restores gene names altered by R/Excel;
- multinomially downsamples each cell to 1,000 counts;
- applies `log1p(count / depth * 10000)`;
- uses repeated downsampling and later averages repeats for each biological cell.

Keep `RUN_PREPROCESS = False` only when the generated `.f32` and `.meta.csv` files already exist under `prepared/`. A fresh clone should run preprocessing once.

In [ ]:
RUN_PREPROCESS = False

if RUN_PREPROCESS:
    if rscript is None:
        raise RuntimeError('Rscript was not found. Install R + Seurat or set rscript manually.')
    subprocess.run([
        rscript, str(SCRIPTS / 'prepare_inputs.R'),
        str(DATA), str(PREPARED), '1000', '4', '2', '20260930'
    ], cwd=ROOT, check=True)
else:
    if (PREPARED / 'E4_5_reference.f32').exists():
        print('Using existing prepared inputs:', PREPARED)
    else:
        print('Prepared matrices are not present. Set RUN_PREPROCESS = True for a full rerun; precomputed example results can still be viewed.')

## 2. Confirm the original critic architecture and checkpoint

The scorer imports the exact critic architecture used in the original notebook: the 2,000-gene vector is concatenated with a five-condition one-hot vector, followed by hidden layers 1024 → 512 → 256 → 128 and a scalar output. No sigmoid is used, so the raw scalar must not be interpreted as a 0–1 probability.

In [ ]:
import torch
sys.path.insert(0, str(SCRIPTS))
from score_critic import LABEL_ORDER, LINEAGE_CONDITIONS, OriginalCritic

genes = [g.strip() for g in (PREPARED / 'genes.txt').read_text(encoding='utf-8').splitlines() if g.strip()]
critic = OriginalCritic(input_dim=len(genes), label_dim=len(LABEL_ORDER))
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
critic.load_state_dict(state)
critic.eval()

print('Conditions:', LABEL_ORDER)
print('Lineage mapping:', LINEAGE_CONDITIONS)
print('Model genes:', len(genes))
print('Critic parameters:', sum(p.numel() for p in critic.parameters()))
critic

## 3. Recalculate natural-calibrated critic distances

The natural cells are split by biological cell ID within each condition: 70% defines the calibration distribution and 30% is retained for the internal holdout score. All blastoid cells are excluded from GAN training and from natural calibration.

Set `RUN_SCORING = True` to regenerate all tables. With 2,000 bootstrap iterations this can take several minutes; otherwise the notebook reads the existing result files.

In [ ]:
RUN_SCORING = False
BOOTSTRAP_ITERATIONS = 2000
SEED = 20261006

if RUN_SCORING:
    subprocess.run([
        sys.executable, str(SCRIPTS / 'score_critic.py'),
        '--data-dir', str(PREPARED),
        '--checkpoint', str(CHECKPOINT),
        '--training-csv', str(DATA / 'E450_expression.csv'),
        '--output-dir', str(RESULTS),
        '--seed', str(SEED),
        '--bootstrap', str(BOOTSTRAP_ITERATIONS),
    ], cwd=ROOT, check=True)
    ACTIVE_RESULTS = RESULTS
else:
    print('Using available results:', ACTIVE_RESULTS)

## 4. Inspect condition-specific and overall rankings

The table below contains the point estimate and the cell-bootstrap 95% CI. For a blastoid dataset, PE-like cells are evaluated independently against Pre_VE and PaE, while TE-like cells are evaluated independently against mTE and pTE. Overall scores are only reported for datasets containing EPI-, PE- and TE-like populations.

In [ ]:
condition_scores = pd.read_csv(ACTIVE_RESULTS / 'condition_and_lineage_distances.csv')
condition_table = (
    condition_scores.query("level == 'condition'")
    [['condition', 'dataset', 'n_cells', 'distance', 'ci_low', 'ci_high']]
    .sort_values(['condition', 'distance'])
    .reset_index(drop=True)
)
condition_table

In [ ]:
overall = pd.read_csv(ACTIVE_RESULTS / 'overall_distances.csv').sort_values('distance')
overall.style.format({'distance': '{:.3f}', 'ci_low': '{:.3f}', 'ci_high': '{:.3f}'})

## 5. Draw the violin figures

The first figure summarizes EPI, PE, TE and overall distances. The second figure shows Pre_VE, PaE, mTE and pTE separately, with one color per blastoid dataset. Violin widths show bootstrap distributions; white points are point estimates and horizontal bars are 95% CIs.

In [ ]:
RUN_PLOTS = False

plot_scripts = [
    SCRIPTS / 'plot_summary.R',
    SCRIPTS / 'plot_four_conditions.R',
    SCRIPTS / 'plot_mismatch_heatmap.R',
]
if RUN_PLOTS:
    if rscript is None:
        raise RuntimeError('Rscript was not found.')
    for script in plot_scripts:
        subprocess.run([rscript, str(script), str(RESULTS)], cwd=ROOT, check=True)
    ACTIVE_RESULTS = RESULTS
else:
    print('Using existing figure files.')

In [ ]:
figure = ACTIVE_RESULTS / 'figure_original_critic_four_conditions_violin.png'
display(Image(filename=str(figure), width=1100))

## 6. Cell-type mismatch challenge

For each annotated blastoid lineage, the exact same cells are evaluated under all five critic labels. Black-bordered heatmap cells are biologically matched labels; unbordered cells are deliberately mismatched labels. Lower distance means closer to the corresponding natural condition. A useful condition-specific critic should generally assign lower distances to labels within the annotated lineage than to cross-lineage labels.

For natural E4.5 holdout cells, `mismatch_penalty = wrong-label distance − correct-label distance`; a positive value means that the wrong label makes the natural cell population less similar to its reference.

In [ ]:
mismatch = pd.read_csv(ACTIVE_RESULTS / 'blastoid_celltype_mismatch_matrix.csv')
mismatch[['dataset', 'annotated_lineage', 'supplied_condition', 'within_annotated_lineage', 'distance', 'ci_low', 'ci_high']].head(15)

In [ ]:
heatmap = ACTIVE_RESULTS / 'figure_original_critic_celltype_mismatch_heatmap.png'
display(Image(filename=str(heatmap), width=1200))

In [ ]:
natural_mismatch = pd.read_csv(ACTIVE_RESULTS / 'natural_e45_celltype_mismatch_matrix.csv')
natural_mismatch[['true_condition', 'supplied_condition', 'mismatch_penalty', 'ci_low', 'ci_high', 'probability_penalty_positive']]

## 7. Output files

Important outputs are:

- `cell_critic_outputs.csv`: per-cell critic outputs under all five labels;
- `condition_and_lineage_distances.csv`: condition- and lineage-level calibrated distances and CIs;
- `overall_distances.csv`: complete-blastoid overall ranking;
- `natural_e45_celltype_mismatch_matrix.csv`: natural holdout wrong-label test;
- `blastoid_celltype_mismatch_matrix.csv`: complete blastoid mismatch test;
- `*_bootstrap.csv`: bootstrap distributions used for violin plots and confidence intervals.

This workflow changes only the interpretation and calibration of the critic outputs. It does not retrain the original GAN or overwrite `critic.pth` / `generator.pth`.

In [ ]:
pd.DataFrame([
    {'file': path.name, 'size_MB': path.stat().st_size / 1024**2}
    for path in sorted(ACTIVE_RESULTS.glob('*')) if path.is_file()
]).sort_values('file').reset_index(drop=True)